# MoonScout — project source notebook

Edit and save this notebook, then run `python run_notebooks.py` from the project folder.
This notebook generates `core.py`, `rasters.py`, `app.py`, and `test_app.py`. The small `run_notebooks.py` exporter remains separate so it can read the notebook without executing it.

**Do not Run All.** Source cells define separate Python modules, not one shared notebook program. The launcher exports them in document order within each module, then runs Streamlit in its own process. Kernel state and saved outputs are ignored.

Each source cell has a visible `# export: module.py` line and a matching `export:module` metadata tag. Duplicate an existing cell in the same section when adding code, preserving both. Untagged code, unknown targets, mismatched markers and syntax errors stop export before any Python files are changed.

The original four notebooks are backed up outside this deliverable. Edit only this notebook for application changes. Generated Python files are overwritten on each launch; commit them with the saved notebook if using GitHub.

Commands:
- `python run_notebooks.py --check`: validate and regenerate all four Python modules without starting the app.
- `python run_notebooks.py --test`: regenerate modules and run tests using the real source cache.
- `python run_notebooks.py --port 8503`: regenerate modules and launch Streamlit.

Stop the running server with Ctrl+C before exporting changes. Save and restart to apply notebook edits. The `.data/` cache and `.streamlit/` theme remain unchanged.


## Analysis, query planning and K-means

Export target: `core.py`. 

This file contains the analysis and query logic. Standardizes TiO₂ and OMAT, fits K-means, calculates cluster separation and candidate scores, interprets requests through Ollama or the offline parser, validates search plans, filters and ranks samples, and generates factual result summaries.

Geographic model additions: cell 3 defines the documented exposure function and provenance; cell 4 attaches it to valid samples; cells 5–8 support optional exposure queries, ranking, and explanations. Exposure is excluded from K-means and the balanced score.

GMM comparison: cell 4 fits both methods on identical standardized features and retains soft membership probabilities; select_terrain switches labels without changing rankings.

Keep imports and definitions in execution order within this section.


In [ ]:
# export: core.py

# Import JSON/regex helpers, numerical tables, HTTP requests, and scikit-learn tools
# Used for deterministic analysis and OpenAI query planning.
import json
import os
import numpy as np
import pandas as pd
import requests
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.mixture import GaussianMixture

# DEFAULT defines the initial search region in degrees, minimum TiO2 in wt%,
# maximum shortlist size, and ranking priority. Callers copy it before editing.
DEFAULT = dict(west=-60., east=60., south=-40., north=40., minimum_tio2=2., top_n=8, priority='balanced')

# OpenAI configuration: an environment variable can select another model that
# supports Responses API Structured Outputs. The app also reads Streamlit secrets.
OPENAI_MODEL = os.getenv('OPENAI_MODEL', 'gpt-4.1-mini')

# Optional code-based key location requested by the project owner. Leave empty
# for a published repository and configure OPENAI_API_KEY in Streamlit secrets.
OPENAI_API_KEY = ''

# A strict envelope lets the model explicitly decline unsupported requests.
# Every plan field is required; validate_plan still enforces numeric/business rules.
PLAN_SCHEMA = {
    'type': 'object', 'additionalProperties': False,
    'properties': {
        'supported': {'type': 'boolean'},
        'plan': {
            'type': 'object', 'additionalProperties': False,
            'properties': {
                **{key: {'type': 'number'} for key in
                   ['west', 'east', 'south', 'north', 'minimum_tio2']},
                'top_n': {'type': 'integer'},
                'priority': {'type': 'string', 'enum': ['balanced', 'tio2', 'maturity', 'exposure']},
            },
            'required': list(DEFAULT),
        },
    },
    'required': ['supported', 'plan'],
}

# Geographic exposure model configuration. f=0.20 is a scenario assumption
# (5.906 days per 29.53-day orbit), not a fitted or observed shielding fraction.
EXPOSURE_SHIELDED_FRACTION = 0.20
EXPOSURE_MODEL = {
    'name': 'Flat-surface normalized geographic solar-wind exposure',
    'version': '1.0',
    'source': 'https://doi.org/10.3390/rs12203350',
    'equation': '4, flat-surface simplification; divided by equatorial farside value',
    'shielded_fraction': EXPOSURE_SHIELDED_FRACTION,
    'synodic_month_days': 29.53,
    'units': 'dimensionless, 0 to 1',
    'longitude_convention': 'degrees east; 0=sub-Earth meridian, +/-180=farside',
    'assumptions': ['constant incident solar wind', 'complete shielding during fraction f',
                    'spherical Moon; zero axial tilt; constant local incidence factor'],
    'excludes': ['OMNI or other measured particle data', 'terrain and local magnetic shielding',
                 'soil exposure age, retention and saturation', 'helium-3 concentration'],
}


def geographic_exposure(latitude, longitude, shielded_fraction=EXPOSURE_SHIELDED_FRACTION):
    """Return relative monthly exposure (0..1), not flux or He-3 abundance.

    Adapted from Shukla et al. (2020), Eq. 4. A constant surface-incidence
    factor and F0 cancel on normalization by the equatorial farside maximum.
    Scalars and broadcast-compatible arrays are accepted; angles are degrees.
    """
    # Bound the physical domain before evaluating the three longitude regimes.
    if (isinstance(shielded_fraction, (bool, np.bool_)) or
            not np.isscalar(shielded_fraction)):
        raise ValueError('Shielded fraction must be a finite number from 0 to 0.5.')
    f = float(shielded_fraction)
    if not np.isfinite(f) or not 0 <= f <= 0.5:
        raise ValueError('Shielded fraction must be a finite number from 0 to 0.5.')
    lat, lon = np.broadcast_arrays(np.asarray(latitude, dtype=float),
                                  np.asarray(longitude, dtype=float))
    if not np.all(np.isfinite(lat)) or not np.all(np.isfinite(lon)) or np.any(np.abs(lat) > 90):
        raise ValueError('Coordinates must be finite and latitude must be within +/-90 degrees.')
    # Wrap either east-longitude convention to [-180,180); east/west exposure
    # is symmetric in this idealized model. This also joins the dateline cleanly.
    phi = np.deg2rad(np.abs((lon + 180.) % 360. - 180.))
    edge_near, edge_far = np.pi * (0.5 - f), np.pi * (0.5 + f)
    # Integrate the sunward projected area outside the completely shielded arc:
    # central nearside, transition toward the limbs, then unaffected farside.
    longitude_factor = np.where(
        phi <= edge_near,
        2. + np.sin(phi - f*np.pi) - np.sin(phi + f*np.pi),
        np.where(phi < edge_far, 1. + np.sin(phi - f*np.pi), 2.))
    # Division by 2 fixes a global reference, independent of the selected region.
    # Clip roundoff only; the mathematical model already lies between zero and one.
    return np.clip(np.cos(np.deg2rad(lat)) * longitude_factor / 2., 0., 1.)

In [ ]:
# export: core.py

# Both methods use the SAME standardized features and rows. Geographic exposure
# stays outside clustering so this experiment isolates the choice of algorithm.
TERRAIN_FEATURES = ['tio2', 'omat']


def sampled_silhouette(x, labels, indices):
    """Measure hard-label separation on shared rows, or None if undefined."""
    # A fixed set of indices makes the two scores comparable. A model may assign
    # only one group in this subset, or one group per row; silhouette then has no
    # mathematical definition. Report unavailable rather than inventing a zero.
    sampled_labels = np.asarray(labels)[indices]
    if not 1 < len(np.unique(sampled_labels)) < len(indices):
        return None
    return float(silhouette_score(x[indices], sampled_labels))


def fit_terrain(df, k=4):
    """Fit K-means plus a GMM candidate; retain the existing two-value API.

    The returned frame keeps K-means as its active `cluster` column. Both sets
    of labels and all GMM membership probabilities are retained for comparison.
    JSON-serializable diagnostics/configuration live in attrs['terrain_comparison'].
    """
    df = df.copy().reset_index(drop=True)
    if isinstance(k, bool) or not isinstance(k, (int, np.integer)) or k < 2:
        raise ValueError('Terrain comparison requires an integer group count of at least 2.')
    features = df[TERRAIN_FEATURES].to_numpy(dtype=float)
    if not np.isfinite(features).all():
        raise ValueError('Terrain features must contain only finite values.')
    if len(df) <= k or len(df[TERRAIN_FEATURES].drop_duplicates()) < k:
        raise ValueError(f'Need more than {k} rows and at least {k} distinct TiO2/OMAT pairs.')

    # Fit the scaler once, then pass exactly the same matrix to both methods.
    # This prevents unit differences (wt% versus OMAT) from dominating distances.
    scaler = StandardScaler().fit(features)
    x = scaler.transform(features)
    km = KMeans(n_clusters=k, random_state=530, n_init=10).fit(x)
    df['kmeans_cluster'] = km.labels_.astype(str)
    df['cluster'] = df['kmeans_cluster']
    df['terrain_model'] = 'K-means'
    df['site_id'] = [f'L{i:05d}' for i in range(len(df))]
    df['solar_wind_exposure'] = geographic_exposure(df.latitude, df.longitude)
    # Ranking is deliberately independent of grouping. Switching methods must
    # not change candidate IDs/order, proxy scores, or geographic exposure.
    df['score'] = 100*(.65*df.tio2.rank(pct=True)+.35*(-df.omat).rank(pct=True))

    # Reuse a deterministic subset for both silhouette evaluations. The report
    # records the sample size and seed so evaluation can be reproduced.
    indices = np.random.RandomState(530).permutation(len(df))[:min(1500,len(df))]
    km_sil = sampled_silhouette(x, km.labels_, indices)
    report = dict(features=TERRAIN_FEATURES.copy(), groups=int(k), seed=530,
                  n_init=10, samples=len(df), silhouette_samples=len(indices),
                  scaler_mean=scaler.mean_.tolist(), scaler_scale=scaler.scale_.tolist(),
                  kmeans_silhouette=km_sil, gmm_silhouette=None,
                  adjusted_rand=None, gmm_available=False, gmm_status='Not fitted',
                  covariance_type='full', reg_covar=1e-6, max_iter=300,
                  gmm_bic=None, gmm_aic=None, gmm_iterations=None,
                  mean_max_membership=None)
    # Full covariance lets each component have its own tilted ellipse in feature
    # space. Small diagonal regularization protects nearly singular covariances.
    # Ten starts reduce dependence on initialization; the seed fixes repeatability.
    try:
        gmm = GaussianMixture(n_components=k, covariance_type='full', n_init=10,
                              random_state=530, reg_covar=1e-6, max_iter=300).fit(x)
        if not gmm.converged_:
            raise ValueError('GMM did not converge within 300 iterations.')
        probabilities = gmm.predict_proba(x)
        labels = probabilities.argmax(axis=1)
        df['gmm_cluster'] = labels.astype(str)
        # Membership is conditional on the fitted Gaussian model. It is NOT a
        # calibrated probability of a geological class, useful deposit, or He-3.
        df['gmm_max_membership'] = probabilities.max(axis=1)
        for component in range(k):
            df[f'gmm_p_{component}'] = probabilities[:,component]
        report.update(gmm_available=True, gmm_status='Converged',
                      gmm_iterations=int(gmm.n_iter_),
                      gmm_silhouette=sampled_silhouette(x,labels,indices),
                      adjusted_rand=float(adjusted_rand_score(km.labels_,labels)),
                      mean_max_membership=float(probabilities.max(axis=1).mean()),
                      gmm_bic=float(gmm.bic(x)), gmm_aic=float(gmm.aic(x)))
    except (ValueError, np.linalg.LinAlgError) as error:
        # GMM is experimental: a failed fit must not disable the working K-means
        # workflow or pretend that K-means labels are GMM output.
        report['gmm_status'] = f'Unavailable: {error}'
    df.attrs['terrain_comparison'] = report
    return df, km_sil


def select_terrain(df, method='K-means'):
    """Return a view copy with the chosen label column; never refit or rerank."""
    # A single alias keeps map colors, popups, explanations, and CSV exports in
    # agreement, while preserving both original label columns for comparison.
    if method not in ['K-means','GMM']:
        raise ValueError('Choose K-means or GMM.')
    if method == 'GMM' and not df.attrs['terrain_comparison']['gmm_available']:
        raise ValueError('GMM is unavailable; inspect the comparison report.')
    selected = df.copy()
    selected['cluster'] = selected['kmeans_cluster' if method=='K-means' else 'gmm_cluster']
    selected['terrain_model'] = method
    return selected

In [ ]:
# export: core.py

# validate_plan(plan) checks the complete query schema and returns the same plan
# on success; invalid fields, bounds, counts, or priorities raise ValueError.
def validate_plan(plan):
    if set(plan) != set(DEFAULT):
        raise ValueError('The query plan has unexpected or missing fields.')
    
    # Reject booleans, nonnumeric values, NaN, and infinity in numeric fields.
    for key in ['west','east','south','north','minimum_tio2','top_n']:
        if isinstance(plan[key], bool) or not isinstance(plan[key], (float,int)) or not np.isfinite(plan[key]):
            raise ValueError(f'Invalid numeric value for {key}.')
    
    # Require ordered lunar bounds; a single region cannot cross the date line.
    if not -180<=plan['west']<plan['east']<=180 or not -90<=plan['south']<plan['north']<=90:
        raise ValueError('Use ordered lunar bounds: west < east, south < north. Split dateline-crossing regions.')
    
    # Constrain the TiO2 threshold and require a whole-number shortlist size.
    if not 0<=plan['minimum_tio2']<=15 or not 1<=plan['top_n']<=50 or int(plan['top_n'])!=plan['top_n']:
        raise ValueError('TiO2 must be 0–15 wt%; candidate count must be an integer from 1–50.')
    if plan['priority'] not in ['balanced','tio2','maturity','exposure']:
        raise ValueError('Unsupported ranking priority.')
    return plan

In [ ]:
# export: core.py

# plan_query sends a question and current constraints to OpenAI and returns a
# validated plan. It never sends raster files, executes model code, or uses an
# offline parser. Failed requests leave state management to the calling UI.
def plan_query(query, current, api_key=None, model=OPENAI_MODEL):
    if not query.strip():
        raise ValueError('Enter a prospecting question.')
    validate_plan(current)
    key = (api_key or os.getenv('OPENAI_API_KEY') or OPENAI_API_KEY).strip()
    if not key:
        raise ValueError('Configure OPENAI_API_KEY in Streamlit secrets or the server environment.')
    system = (
        'Translate lunar prospecting requests into the supplied JSON schema. '
        'Preserve current plan values unless the user changes them. '
        'Bounds are degrees east [-180,180] and north [-90,90]; minimum_tio2 is wt%. '
        'top_n must be 1-50. priority is balanced, tio2, maturity, or exposure. '
        'Lower OMAT means greater optical maturity. Best spots means balanced proxy ranking. '
        'Only geographic bounds, TiO2 thresholds, candidate count, and these ranking priorities are supported. '
        'Requests for highest solar-wind exposure use priority=exposure: this is a modeled relative '
        'geographic exposure index, not measured flux or accumulated dose. '
        'Exposure thresholds and changes to shielding assumptions are unsupported. '
        'For requests requiring helium abundance, measured solar wind, time-dependent flux, slope, landing safety, other elements, '
        'mining feasibility, or any unavailable constraint, set supported=false and return the current plan unchanged. '
        'Never silently drop an unsupported constraint or invent measurements. '
        'For supported requests set supported=true. Current plan: ' + json.dumps(current)
    )
    # REST avoids an additional SDK dependency. Disable response storage and bound
    # connection/read waits. Credentials stay in the server-side Authorization header.
    try:
        response = requests.post(
            'https://api.openai.com/v1/responses',
            headers={'Authorization': f'Bearer {key}'},
            json={
                'model': model, 'store': False, 'max_output_tokens': 1200,
                'input': [{'role': 'system', 'content': system},
                          {'role': 'user', 'content': query}],
                'text': {'format': {'type': 'json_schema', 'name': 'lunar_query',
                                    'strict': True, 'schema': PLAN_SCHEMA}},
            }, timeout=(10, 90),
        )
    except requests.RequestException:
        raise ValueError('OpenAI could not be reached. Please retry shortly.') from None
    # Do not echo raw provider errors, request headers, or keys into the public UI.
    if response.status_code in (401, 403):
        raise ValueError('OpenAI authentication failed. Check the server API key and project access.')
    if response.status_code == 429:
        raise ValueError('OpenAI quota or rate limit reached. Check billing or retry later.')
    if response.status_code != 200:
        raise ValueError('OpenAI request failed. Check the configured model and service availability.')
    try:
        payload = response.json()
        if payload.get('status') != 'completed':
            raise ValueError('incomplete')
        # Responses may contain non-message items. Read only assistant text and
        # reject refusals instead of treating them as a valid query plan.
        content = [part for item in payload.get('output', [])
                   if item.get('type') == 'message' for part in item.get('content', [])]
        if any(part.get('type') == 'refusal' for part in content):
            raise ValueError('refused')
        envelope = json.loads(''.join(part['text'] for part in content
                                      if part.get('type') == 'output_text'))
        if set(envelope) != {'supported', 'plan'} or type(envelope['supported']) is not bool:
            raise ValueError('invalid envelope')
        plan = validate_plan(envelope['plan'])
    except (ValueError, TypeError, KeyError, AttributeError):
        raise ValueError('OpenAI did not return a complete valid query plan. Please rephrase or retry.') from None
    if not envelope['supported']:
        raise ValueError('This request requires unavailable data or constraints. Use lunar bounds, TiO2, optical maturity, modeled exposure ranking, and candidate count.')
    return plan

In [ ]:
# export: core.py

# rank_sites(df, plan) returns up to top_n samples within the inclusive lunar
# bounds and above the TiO2 threshold, after validating the plan.
def rank_sites(df, plan):
    validate_plan(plan)
    eligible = df[df.longitude.between(plan['west'],plan['east']) & df.latitude.between(plan['south'],plan['north']) & (df.tio2>=plan['minimum_tio2'])].copy()
    
    # Balanced/TiO2/exposure rankings sort highest first; maturity sorts lowest OMAT first.
    # Use site_id as a stable secondary sort key when primary values tie.
    key, ascending = {'balanced':('score',False),'tio2':('tio2',False),'maturity':('omat',True),'exposure':('solar_wind_exposure',False)}[plan['priority']]
    return eligible.sort_values([key,'site_id'],ascending=[ascending,True]).head(int(plan['top_n']))

In [ ]:
# export: core.py

# explain(result, plan) builds a deterministic text summary from the actual
# shortlist and its leading sample; no LLM or additional measurements are used.
def explain(result, plan):
    
    # Report an empty shortlist explicitly rather than silently relaxing constraints.
    if result.empty:
        return 'No samples meet these constraints. Expand the region or lower the minimum TiO2; constraints have not been relaxed automatically.'
    
    # Use the first ranked row and state the limits of these prospecting proxies.
    row = result.iloc[0]
    prefix = 'Imported raster samples: '
    return (f'{prefix}{len(result)} candidates ranked by {plan["priority"]}. '
            f'Leading sample {row.site_id} is at {row.latitude:.2f}°N, {row.longitude:.2f}°E, '
            f'with TiO2 {row.tio2:.2f} wt%, OMAT {row.omat:.3f}, and {row.terrain_model} terrain group {row.cluster}. '
            f'Modeled relative solar-wind exposure is {row.solar_wind_exposure:.3f} on a 0-1 scale '
            f'(shielded fraction {EXPOSURE_SHIELDED_FRACTION:.2f}; not measured flux). '
            'These proxies support follow-up investigation; they do not measure helium-3 or establish landing suitability.')

## Approved source downloads and sampling

Export target: `rasters.py`. 

This file automatically downloads and caches the approved LROC and USGS datasets. Reads their lunar coordinate systems, samples matching locations, handles missing or invalid values, and records source provenance. It supplies the measurements used by the rest of the app.

Keep imports and definitions in execution order within this section.


In [ ]:
# export: rasters.py

# Import filesystem, concurrent download, hashing, HTTP, table, and raster/CRS
# tools. Rasterio reads scientific pixels; Leafmap handles display in app.py.
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import hashlib
import json
import os
import threading
import requests
import numpy as np
import pandas as pd
import rasterio
from pyproj import CRS, Transformer

# SOURCE_PAGES records the two approved catalog pages used in provenance.
SOURCE_PAGES = {
 'tio2':'https://data.lroc.im-ldi.com/lroc/view_rdr/WAC_TIO2',
 'omat':'https://astrogeology.usgs.gov/search/map/kaguya_lunar_multiband_imager_mi_derived_optical_maturity_omat_50n50s_512ppd_59mpp',
}

# BASE is the publisher archive directory for the scientific WAC TiO2 tiles.
BASE='https://pds.lroc.im-ldi.com/data/LRO-L-LROC-5-RDR-V1.0/LROLRC_2001/DATA/SDP/WAC_TIO2/'

# OMAT_URL selects the publisher-provided 4-pixels-per-degree overview of
# the approved OMAT product, rather than downloading its full 512-ppd raster.
OMAT_URL='https://asc-pds-services.s3.us-west-2.amazonaws.com/mosaic/Lunar_MI_mineral_maps/4ppd/Lunar_Kaguya_MIMap_MineralDeconv_OpticalMaturityIndex_50N50S.tif'

# TILES enumerates eight scientific IMG products: four longitude sectors
# for each lunar hemisphere; these are not the browse-image products.
TILES=[f'WAC_TIO2_E350{hem}{lon}.IMG' for hem in ['N','S'] for lon in ['0450','1350','2250','3150']]

# CACHE uses the launcher-provided data directory, or a .data folder next
# to the generated rasters.py file when no environment override is set.
CACHE=Path(os.environ.get('MoonScout_DATA_DIR', str(Path(__file__).parent/'.data')))

# LOCK serializes source loading within this Python process; it is not
# a filesystem lock shared by multiple running app processes.
LOCK=threading.Lock()

# VERSION identifies the sampled-table cache format; change it when cached
# samples must be rebuilt after changes to sampling or filtering.
VERSION=1

In [ ]:
# export: rasters.py

# download(url, name) returns a cached raster path and provenance receipt,
# or downloads and validates the file before replacing the cached copy.
# The caller supplies the fixed publisher URLs and creates CACHE first.
def download(url,name):
    path=CACHE/name
    receipt=path.with_suffix(path.suffix+'.json')
    
    # Reuse a file only when its receipt URL and recorded byte count match.
    # This fast cache check does not recompute the saved SHA-256 digest.
    if path.exists() and receipt.exists():
        meta=json.loads(receipt.read_text())
        if meta.get('source_url')==url and meta.get('bytes')==path.stat().st_size:
            return path,meta
    
    # Stream to a .part file while computing byte count and a local SHA-256
    # fingerprint; this digest is not an independently verified publisher checksum.
    temporary=path.with_suffix(path.suffix+'.part')
    digest=hashlib.sha256(); count=0
    with requests.get(url,stream=True,timeout=(15,90)) as response:
        response.raise_for_status()
        with temporary.open('wb') as out:
            for chunk in response.iter_content(1024*1024):
                out.write(chunk); digest.update(chunk); count+=len(chunk)
        expected=response.headers.get('Content-Length')
        if expected and int(expected)!=count: raise ValueError(f'Incomplete source download: {name}')
        meta={'source_url':url,'resolved_url':response.url,'bytes':count,'sha256':digest.hexdigest()}
    
    # Check that the downloaded product has one band and a coordinate system
    # before atomically replacing the final raster path and writing its receipt.
    with rasterio.open(temporary) as ds:
        if ds.count!=1 or not ds.crs: raise ValueError(f'Invalid scientific raster: {name}')
    os.replace(temporary,path)
    receipt.write_text(json.dumps(meta,indent=2))
    return path,meta

In [ ]:
# export: rasters.py

# sample(ds, lon, lat) returns scaled nearest-pixel values for matching lunar
# longitude/latitude arrays; uncovered or masked locations remain NaN.
def sample(ds,lon,lat):
    
    # Check for a Moon-sized ellipsoid, then transform from this raster
    # CRS's own geographic coordinates instead of assuming an Earth datum.
    crs=CRS.from_user_input(ds.crs)
    if not 1700000<crs.ellipsoid.semi_major_metre<1800000:
        raise ValueError('Publisher raster does not have a lunar CRS.')
    transformer=Transformer.from_crs(crs.geodetic_crs,crs,always_xy=True,force_over=True)
    values=np.full(len(lon),np.nan)
    
    # Try equivalent longitude ranges to accommodate publisher tiles using
    # different wrapping conventions, filling only locations still missing.
    for shift in [0,360,-360]:
        x,y=transformer.transform(lon+shift,lat); x=np.asarray(x); y=np.asarray(y)
        inside=(x>=ds.bounds.left)&(x<ds.bounds.right)&(y>ds.bounds.bottom)&(y<=ds.bounds.top)&np.isnan(values)
        ids=np.flatnonzero(inside)
        if len(ids):
            a=np.ma.vstack(list(ds.sample(zip(x[ids],y[ids]),indexes=1,masked=True))).astype(float)
            
            # Convert masked pixels to NaN and apply the band's scale and offset.
            values[ids]=a.filled(np.nan).ravel()*ds.scales[0]+ds.offsets[0]
    return values

In [ ]:
# export: rasters.py

# load_sources(progress) returns a real sample DataFrame and provenance list.
# The optional progress callback receives status strings; its default lambda
# does nothing. Loading and cache writes are serialized within this process.
def load_sources(progress=lambda message: None):
    with LOCK:
        CACHE.mkdir(exist_ok=True)
        saved=CACHE/'samples.json'
        
        # A matching sample-cache version bypasses downloads and sampling, restoring
        # the table attributes used to reconstruct the display grid.
        if saved.exists():
            payload=json.loads(saved.read_text())
            if payload.get('version')==VERSION:
                df=pd.DataFrame(payload['rows']); df.attrs=payload['attrs']
                return df,payload['provenance']
        progress('Downloading scientific LROC tiles and USGS OMAT. First load may take several minutes (about 1.2 GB); later visits use the local cache.')
        
        # Download the eight fixed TiO2 tiles and one OMAT overview with four workers.
        # The lambda unpacks each (URL, filename) job into download; map preserves order.
        jobs=[(BASE+n,n) for n in TILES]+[(OMAT_URL,'omat-4ppd.tif')]
        with ThreadPoolExecutor(max_workers=4) as pool:
            files=list(pool.map(lambda job:download(*job),jobs))
        
        # Define sampling coordinates every two degrees within shared coverage.
        # This creates locations only: all measurement values come from the rasters.
        lon,lat=np.meshgrid(np.arange(-179.,180.,2.),np.arange(49.,-50.,-2.))
        df=pd.DataFrame({'longitude':lon.ravel(),'latitude':lat.ravel()})
        tio=np.full(len(df),np.nan); provenance=[]
        
        # Read each scientific TiO2 tile, retain finite values from 0 to 15 wt%,
        # and record the source receipt, units, resolution, and band calibration.
        for (path,meta),name in zip(files[:-1],TILES):
            progress(f'Sampling {name}')
            with rasterio.open(path) as ds:
                if ds.dtypes[0]!='float32': raise ValueError('Expected scientific float32 WAC product, not browse imagery.')
                a=sample(ds,df.longitude.to_numpy(),df.latitude.to_numpy())
                good=np.isfinite(a)&(a>=0)&(a<=15)
                tio[good]=a[good]
                provenance.append({**meta,'source_page':SOURCE_PAGES['tio2'],'product':name,'units':'wt%','native_resolution':'about 400 m/pixel','scale':ds.scales[0],'offset':ds.offsets[0]})
        
        # Sample OMAT at the same lunar coordinates so each row pairs the two proxies.
        path,meta=files[-1]
        with rasterio.open(path) as ds:
            df['omat']=sample(ds,df.longitude.to_numpy(),df.latitude.to_numpy())
        df['tio2']=tio
        
        # Keep only overlapping finite measurements with OMAT in [0, 1]. Missing
        # measurements are excluded, never interpolated or filled with substitute data.
        valid=np.isfinite(df[['tio2','omat']]).all(axis=1)&df.omat.between(0,1)
        df=df[valid].reset_index(drop=True)
        if len(df)<5: raise ValueError('Source products yielded too few valid overlapping samples.')
        
        # Preserve the full coordinate axes for map reconstruction, including holes
        # left by invalid samples, and describe the sampling choices in provenance.
        df.attrs={'grid_longitudes':lon[0].tolist(),'grid_latitudes':lat[:,0].tolist()}
        provenance.append({**meta,'source_page':SOURCE_PAGES['omat'],'product':'Kaguya MI Optical Maturity Index 50N50S','resolution':'Publisher-provided 4 ppd overview of the 512 ppd dataset','credit':'JAXA/MoonScout; Lemelin et al. (2016)'})
        provenance.append({'sampling':'Nearest pixel at 2-degree grid centers; no missing values filled','valid_samples':len(df),'excluded_samples':int((~valid).sum()),'coverage':'180W–180E, 50S–50N','quality_filter':'TiO2 0–15 wt%; OMAT 0–1'})
        
        # Serialize rows, grid attributes, and provenance together; reject NaN in
        # JSON and replace the sample-cache file only after writing completes.
        payload={'version':VERSION,'rows':df.to_dict('records'),'attrs':df.attrs,'provenance':provenance}
        tmp=saved.with_suffix('.tmp'); tmp.write_text(json.dumps(payload,allow_nan=False)); os.replace(tmp,saved)
        return df,provenance

## Streamlit and Leafmap interface

Export target: `app.py`. 

This file builds the Streamlit interface: navigation, question input, search controls, Leafmap display, candidate table, downloads, and methods explanation. Coordinates data loading and query execution, and stores the current settings and conversation in session state.

Cell 15 adds the modeled exposure layer, ranking option, shortlist column, popup values, export provenance, and Methods explanation.

GMM interface: cell 15 adds the sidebar selector and Methods comparison, including profiles, plots, diagnostics, ambiguous assignments and downloads.

Keep imports and definitions in execution order within this section.


In [ ]:
# export: app.py

# This cell defines the Streamlit page at module scope: Streamlit reruns it
# on interaction, while session_state preserves loaded data and user settings.
import json
import os
import numpy as np
import streamlit as st
import leafmap.foliumap as leafmap
import folium
# Altair (bundled with Streamlit) gives both comparison plots identical axes.
import altair as alt
from core import DEFAULT, fit_terrain, plan_query, rank_sites, explain, OPENAI_API_KEY, OPENAI_MODEL, EXPOSURE_MODEL, select_terrain
from rasters import load_sources, SOURCE_PAGES

# Configure the browser page and inject the interface's compact visual styling.
st.set_page_config(page_title='MoonScout | Lunar prospecting', page_icon='🌘', layout='wide')
st.markdown('''<style>.block-container{padding-top:2rem}h1{letter-spacing:-1.8px} div[data-testid="stMetric"]{background:#151e2d;padding:12px;border-radius:12px} div[data-testid="stMetricValue"]{font-size:1.4rem} .eyebrow{color:#76e0c5;letter-spacing:3px;font-size:12px}</style>''',unsafe_allow_html=True)

# Invalidate old fitted tables so every session gains both model labels and diagnostics.
if st.session_state.get('data_policy_version') != 5:
    st.session_state.clear()
    st.session_state.update(data_policy_version=5, data=None, silhouette=None, provenance=[], plan=dict(DEFAULT), history=[])

# Read server configuration without displaying or storing credentials in chat.
# Streamlit secrets take precedence over environment variables and code defaults.
def server_setting(name, default=''):
    try:
        value = st.secrets.get(name)
    except FileNotFoundError:
        value = None
    return str(value or os.getenv(name) or default).strip()

api_key = server_setting('OPENAI_API_KEY', OPENAI_API_KEY)
model = server_setting('OPENAI_MODEL', OPENAI_MODEL)

# Navigation and query-engine status. All questions use OpenAI.
with st.sidebar:
    st.markdown('### 🌘 MoonScout')
    st.caption('LUNAR PROSPECTING WORKSPACE')
    page = st.radio('Workspace', ['Explore','Methods & project'],label_visibility='collapsed')
    # Switching models only selects existing labels; no model is refitted on
    # each widget interaction. K-means remains the default and always works
    # even if the experimental GMM cannot converge on a particular dataset.
    available = ['K-means']
    if st.session_state.data is not None and st.session_state.data.attrs['terrain_comparison']['gmm_available']:
        available.append('GMM')
    terrain_method = st.selectbox('Terrain model', available, key='terrain_method')
    st.caption('Compare both models in Methods & project. Ranking is unchanged.')
    st.divider()
    st.caption('QUERY ENGINE')
    st.caption(f'OpenAI · {model}')
    if not api_key:
        st.warning('Set OPENAI_API_KEY in the app secrets to enable questions.')
    st.divider()
    st.write('No dataset loaded' if st.session_state.data is None else 'Source raster data')
    if st.session_state.data is not None:
        st.caption(f'{len(st.session_state.data):,} valid samples · 4 terrain groups')

# Display the shared title and approved-source description on both pages.
st.title('Find your next lunar lead.')
st.caption('Compare titanium abundance, optical maturity, and modeled solar-wind exposure. Build a shortlist for further investigation.')
st.caption('Measured-source layers: LROC WAC TiO₂ and USGS Kaguya MI OMAT. Solar-wind exposure is a geographic model.')

# Explore loads real source data once per session, fits terrain groups,
# and stores data, silhouette, and provenance for later reruns.
if page == 'Explore':
    if st.session_state.data is None:
        try:
            with st.status('Loading approved lunar datasets…', expanded=True) as status:
                data, provenance = load_sources(st.write)
                clustered, silhouette = fit_terrain(data)
                st.session_state.update(data=clustered, silhouette=silhouette, provenance=provenance)
                status.update(label='Source datasets loaded', state='complete', expanded=False)
            st.rerun()
        
        # A loading failure stops the page and offers retry; no substitute data is used.
        except Exception as error:
            st.error(f'The approved source datasets could not be loaded: {error}')
            st.info('No substitute data is used. Check the server connection and retry.')
            if st.button('Retry source loading'):
                st.rerun()
            st.stop()
    st.caption('Automatically loaded • LROC scientific abundance tiles + USGS OMAT 4 ppd overview • 2° sampling grid')
    
    # Read the current plan, then collect region and ranking edits in a form
    # so they are validated and applied together on submission.
    # Apply one model selection consistently to every Explore output.
    active_data=select_terrain(st.session_state.data, terrain_method)
    plan=st.session_state.plan
    with st.expander('Region & ranking controls',expanded=False):
        with st.form('controls'):
            cols=st.columns(4)
            values={key:cols[i].number_input(key.title(),value=float(plan[key])) for i,key in enumerate(['west','east','south','north'])}
            cols=st.columns(3)
            values['minimum_tio2']=cols[0].number_input('Minimum TiO₂ (wt%)',0.,15.,float(plan['minimum_tio2']))
            values['top_n']=cols[1].number_input('Candidates',1,50,int(plan['top_n']))
            values['priority']=cols[2].selectbox('Rank by',['balanced','tio2','maturity','exposure'],index=['balanced','tio2','maturity','exposure'].index(plan['priority']))
            if st.form_submit_button('Apply constraints'):
                from core import validate_plan
                try:
                    st.session_state.plan=validate_plan(values)
                    st.rerun()
                except ValueError as error:
                    st.error(str(error))
    
    # Place question controls/history in the left column and the map/table
    # in the right column; all views share the current session plan.
    left,right=st.columns([.85,1.65],gap='large')
    with left:
        st.subheader('01 / Ask & refine')
        st.caption('Try “highest TiO2”, “highest solar wind exposure”, then “top 5”. Follow-ups retain the current region and filters.')
        with st.form('question',clear_on_submit=True):
            question=st.text_area('Prospecting question',placeholder='Find the best spots',height=105)
            run=st.form_submit_button('Find candidates',type='primary', disabled=not api_key)
        
        # On submission, parse and validate the question, rank real samples, and
        # create a grounded summary before committing the new plan and conversation.
        if run:
            try:
                with st.spinner('Plan → validate → retrieve → rank → explain'):
                    new_plan=plan_query(question,st.session_state.plan,api_key=api_key,model=model)
                    results=rank_sites(active_data,new_plan)
                    response=explain(results,new_plan)
                st.session_state.plan=new_plan
                st.session_state.history.append((question,response,f'OpenAI · {model}'))
            except Exception as error:
                st.error(f'Query stopped; previous results retained. {error}')
        
        # Show the four most recent exchanges and identify which query engine ran.
        for question,response,engine in st.session_state.history[-4:]:
            with st.chat_message('user'): st.write(question)
            with st.chat_message('assistant'):
                st.write(response)
                st.caption(engine)
        if st.button('Clear conversation'):
            st.session_state.history=[]
            st.rerun()
        
        # Expose the processing steps and exact current plan for inspection.
        with st.expander('Workflow trace'):
            st.write('1. Parse request into a constrained plan.\n2. Validate bounds and available fields.\n3. Retrieve matching samples.\n4. Rank by proxies and attach fitted terrain groups.\n5. Explain using exact retrieved values.')
            st.json(st.session_state.plan)
    
    # Recompute the shortlist from current constraints and define summary metrics
    # and the selected display field (TiO2, OMAT, cluster ID, or modeled exposure).
    with right:
        result=rank_sites(active_data,st.session_state.plan)
        st.subheader('02 / Survey the candidates')
        a,b,c=st.columns(3)
        a.metric('Shortlisted',len(result))
        b.metric('Top TiO₂',f'{result.tio2.max():.2f} wt%' if len(result) else '—')
        c.metric('Terrain groups',result.cluster.nunique())
        st.caption(f'Terrain labels: {terrain_method}. Numbers and colors are local to each model, not matching geological classes.')
        layer=st.selectbox('Surface layer',['TiO₂ abundance','Optical maturity','Terrain groups','Modeled solar-wind exposure'])
        
        # Simple CRS maps lunar longitude/latitude directly; Earth distance tools are disabled.
        m=leafmap.Map(center=[0,0],zoom=1,tiles=None,crs='Simple',draw_control=False,
                      measure_control=False,fullscreen_control=True,latlon_control=False,
                      search_control=False,scale_control=False)
        
        # Remove any default Earth tile layers that the map constructor may add.
        for key in list(m._children):
            if isinstance(m._children[key],folium.raster_layers.TileLayer): del m._children[key]
        data=active_data
        
        # The surface preview uses all samples inside the geographic bounds, while
        # the shortlist also applies the minimum TiO2 and candidate-count filters.
        region=data[data.longitude.between(st.session_state.plan['west'],st.session_state.plan['east']) & data.latitude.between(st.session_state.plan['south'],st.session_state.plan['north'])]
        field={'TiO₂ abundance':'tio2','Optical maturity':'omat','Terrain groups':'cluster','Modeled solar-wind exposure':'solar_wind_exposure'}[layer]
        
        # Define categorical cluster colors; continuous layers use a separate ramp.
        colors=['#70e1c4','#f5b86b','#9898f0','#eb83ae']
        
        # Reconstruct the imported regular grid as a small RGBA raster overlay.
        if len(region):
            p=st.session_state.plan
            
            # Recover and crop grid axes, ordering latitude north-to-south to match
            # the image overlay's upper origin.
            xs=np.sort(np.asarray(data.attrs.get('grid_longitudes',data.longitude.unique())))
            ys=np.sort(np.asarray(data.attrs.get('grid_latitudes',data.latitude.unique())))[::-1]
            xs=xs[(xs>=p['west'])&(xs<=p['east'])]; ys=ys[(ys>=p['south'])&(ys<=p['north'])]
            if len(xs)>1 and len(ys)>1:
                
                # Initialize transparent RGBA cells; unobserved locations stay transparent.
                # Coordinate-to-index dictionaries place each real sample in its grid cell.
                grid=np.zeros((len(ys),len(xs),4),dtype=np.uint8)
                xidx={x:i for i,x in enumerate(xs)}; yidx={y:i for i,y in enumerate(ys)}
                
                # Scale continuous colors to the visible region's min/max; cluster colors
                # use discrete IDs. The fallback denominator handles a constant-valued region.
                low,high=region[field].astype(float).min(),region[field].astype(float).max()
                # Exposure uses the global 0..1 reference so colors remain comparable
                # across regions; measured layers retain their existing local scaling.
                if field == 'solar_wind_exposure':
                    low, high = 0., 1.
                for row in region.itertuples():
                    value=float(getattr(row,field)); t=(value-low)/(high-low or 1)
                    color=colors[int(row.cluster)] if field=='cluster' else None
                    rgb=[int(color[i:i+2],16) for i in (1,3,5)] if color else [int(35+210*t),int(65+130*t),int(90+30*t)]
                    grid[yidx[row.latitude],xidx[row.longitude]]=[*rgb,210]
                
                # Extend overlay bounds by half a grid spacing so samples sit at cell centers.
                dx=(xs[1]-xs[0])/2; dy=abs(ys[1]-ys[0])/2
                folium.raster_layers.ImageOverlay(grid,bounds=[[ys[-1]-dy,xs[0]-dx],[ys[0]+dy,xs[-1]+dx]],name=layer,origin='upper').add_to(m)
        
        # Add numbered shortlist markers with exact coordinates, proxy values,
        # and terrain labels, then frame the map using the selected bounds.
        for rank,row in enumerate(result.itertuples(),1):
            popup=f'#{rank} · {row.site_id}<br>{row.latitude:.2f}°N, {row.longitude:.2f}°E<br>TiO₂ {row.tio2:.2f} wt%<br>OMAT {row.omat:.3f}<br>Modeled exposure {row.solar_wind_exposure:.3f} (0–1)<br>{row.terrain_model} terrain {row.cluster}'
            folium.Marker([row.latitude,row.longitude],tooltip=f'#{rank} {row.site_id}',popup=popup,
                icon=folium.DivIcon(html=f'<div style="background:#76e0c5;color:#101722;border:2px solid white;border-radius:50%;width:25px;height:25px;text-align:center;font-weight:bold">{rank}</div>')).add_to(m)
        p=st.session_state.plan
        m.fit_bounds([[p['south'],p['west']],[p['north'],p['east']]])
        m.to_streamlit(height=410)
        st.caption('Lunar longitude/latitude · sampled raster preview, not full-resolution imagery. Continuous layers: dark = low, gold = high. Terrain groups: mint 0 · gold 1 · violet 2 · pink 3.')
        if field == 'solar_wind_exposure':
            st.caption('Modeled relative exposure: fixed 0–1 color scale; 1 = equatorial farside reference. Shielded orbit fraction f=0.20. No measured particle flux, terrain correction, or He-3 estimate.')
        elif len(region) and field!='cluster':
            st.caption(f'Visible color range: {region[field].min():.3f} to {region[field].max():.3f} '+('wt%' if field=='tio2' else 'OMAT'))
        
        # Show an explicit empty-result message or offer the ranked table, CSV
        # with source links, and JSON containing the query and provenance.
        if result.empty: st.info(explain(result,p))
        else:
            # Retain both assignments in the shortlist so switching labels is
            # auditable. Only show GMM fields when its fit actually succeeded.
            columns=['site_id','latitude','longitude','tio2','omat','solar_wind_exposure','terrain_model','cluster','kmeans_cluster','score']
            if 'gmm_cluster' in result:
                columns += ['gmm_cluster','gmm_max_membership']
                st.caption('GMM max membership describes group assignment under the model; it is not He-3 confidence.')
            st.dataframe(result[columns].round(3),hide_index=True,width='stretch')
            st.download_button('Download shortlist (CSV)',result.assign(tio2_source=SOURCE_PAGES['tio2'],omat_source=SOURCE_PAGES['omat'],exposure_model_source=EXPOSURE_MODEL['source'],exposure_model_version=EXPOSURE_MODEL['version'],exposure_shielded_fraction=EXPOSURE_MODEL['shielded_fraction']).to_csv(index=False),'lunar-shortlist.csv','text/csv')
            st.download_button('Download query & provenance',json.dumps({'plan':p,'provenance':st.session_state.provenance,'exposure_model':EXPOSURE_MODEL,'terrain_model':terrain_method,'terrain_comparison':active_data.attrs['terrain_comparison']},indent=2,default=str),'provenance.json','application/json')

# The Methods page explains the pipeline, cluster diagnostics, heuristic
# ranking, source limitations, provenance, and local-model setup.
else:
    st.subheader('A prospecting aid, with evidence in view')
    st.write('Designed for exploratory analysts comparing potential lunar prospecting regions. A successful interaction produces a traceable shortlist, not a mining or landing authorization.')
    st.markdown('### How the pieces fit')
    st.code('Question → LLM JSON plan → validation → raster sample retrieval\n                                           ↓\nPublisher rasters → common lunar grid → standardization → K-means / GMM comparison\n                                           ↓\n                     proxy ranking → Leafmap + table + explanation')
    st.markdown('### Compare K-means and GMM')
    st.write('Both methods fit the same standardized TiO₂ and OMAT samples, with four groups, ten starts and seed 530. GMM uses full covariance, allowing tilted elliptical components and soft memberships. Coordinates and modeled exposure are excluded. K-means remains the default; choose Terrain model in the sidebar to change Explore labels.')
    if st.session_state.data is None:
        st.info('Open Explore to load samples and fit the comparison.')
    else:
        data = st.session_state.data
        report = data.attrs['terrain_comparison']
        # None metrics are genuinely undefined, not zero. Both silhouettes use
        # the same standardized rows; ARI compares partitions without matching IDs.
        def metric_text(value):
            """Format optional diagnostics without implying unavailable values are zero."""
            return 'Unavailable' if value is None else f'{value:.3f}'
        a,b,c = st.columns(3)
        a.metric('K-means silhouette',metric_text(report['kmeans_silhouette']))
        b.metric('GMM silhouette',metric_text(report['gmm_silhouette']))
        c.metric('Adjusted Rand agreement',metric_text(report['adjusted_rand']))
        st.caption('Silhouette: higher indicates stronger distance separation of hard labels, not better He-3 prediction. It can favor compact K-means groups. ARI: 1 means identical partitions up to label renaming; near 0 is chance-level agreement. Agreement is not accuracy.')
        st.write('GMM status: '+report['gmm_status'])
        if report['gmm_available']:
            st.caption(f"GMM mean maximum membership: {report['mean_max_membership']:.3f}. BIC: {report['gmm_bic']:.1f}; AIC: {report['gmm_aic']:.1f}. Lower BIC/AIC can compare GMM configurations on the same data and scaling; these are not comparisons against K-means. This screen fits four components only.")
        else:
            st.warning('GMM is unavailable. K-means remains usable; inspect the diagnostics below.')
        # Show group profiles in ORIGINAL units for interpretation, including
        # group sizes. GMM soft means and hard-assignment averages can differ;
        # the table intentionally describes the displayed hard assignments.
        methods = [('K-means','kmeans_cluster')]
        if report['gmm_available']:
            methods.append(('GMM','gmm_cluster'))
        panels=st.columns(len(methods))
        # The plots use exactly the same deterministic rows, fixed axis ranges,
        # and units. Sampling limits browser work without changing the fitted data.
        plot_data=data.sample(n=min(2000,len(data)),random_state=530)
        for panel,(name,label) in zip(panels,methods):
            with panel:
                st.markdown(f'#### {name}')
                profile=data.groupby(label).agg(samples=('site_id','count'),mean_tio2=('tio2','mean'),mean_omat=('omat','mean')).reset_index().rename(columns={label:'group'})
                st.dataframe(profile.round(3),hide_index=True)
                # Fix scales explicitly: independently auto-scaled panels can
                # make the same sample distribution look different by accident.
                chart=alt.Chart(plot_data).mark_circle(size=18,opacity=.45).encode(
                    x=alt.X('tio2:Q',title='TiO₂ (wt%)',scale=alt.Scale(domain=[0,15])),
                    y=alt.Y('omat:Q',title='OMAT',scale=alt.Scale(domain=[0,1])),
                    color=alt.Color(f'{label}:N',title='Group',scale=alt.Scale(
                        domain=['0','1','2','3'],range=['#70e1c4','#f5b86b','#9898f0','#eb83ae'])),
                    tooltip=['site_id','tio2','omat',label]).properties(height=280)
                st.altair_chart(chart,use_container_width=True)
        if report['gmm_available']:
            # A cross-tab reveals how groups split/merge. Comparing numeric IDs
            # directly would be misleading because each algorithm names arbitrarily.
            st.markdown('#### Assignment cross-tab (all samples)')
            import pandas as pd
            st.dataframe(pd.crosstab(data.kmeans_cluster,data.gmm_cluster))
            st.markdown('#### Most ambiguous GMM assignments')
            st.caption('Lowest maximum membership first. These are ambiguous group assignments, not necessarily poor prospecting candidates.')
            probability_columns=[column for column in data if column.startswith('gmm_p_')]
            st.dataframe(data.nsmallest(10,'gmm_max_membership')[['site_id','tio2','omat','gmm_cluster','gmm_max_membership']+probability_columns].round(3),hide_index=True)
        st.caption('Labels/colors do not correspond across models. Both methods partition continuous proxies and may not recover real geological classes. Spatial correlation, feature choice and initialization affect results. No model is selected automatically from these metrics.')
        with st.expander('Comparison settings and diagnostics'):
            st.json(report)
        # Export all rows and posterior columns for independent analysis. Include
        # configuration and source receipts separately so the comparison is repeatable.
        st.download_button('Download terrain comparison (CSV)',data.drop(columns=['cluster','terrain_model']).to_csv(index=False),'terrain-comparison.csv','text/csv')
        st.download_button('Download comparison diagnostics (JSON)',json.dumps({'comparison':report,'source_provenance':st.session_state.provenance},indent=2,default=str),'terrain-comparison.json','application/json')
    st.markdown('### Ranking and limitations')
    st.write('Balanced score = 100 × (0.65 × TiO₂ percentile + 0.35 × inverse-OMAT percentile) within the loaded source dataset. This is an explicit exploratory heuristic, not a trained helium-3 model. Lower OMAT is treated as greater optical maturity, not a direct solar-wind exposure measurement. Pure TiO₂ and maturity rankings are also available. No helium-3 ground truth, slope, landing safety, other elements, extraction economics or direct solar-wind data are included.')
    st.write('WAC abundance values below 2 wt% are below the product detection limit; values above 15 wt% are excluded. OMAT outside 0–1 is excluded as a prototype quality check. Verify product-specific calibration and masks. Grid sampling can miss small high-value regions; nearby shortlist rows are not independent deposits.')
    # Keep model provenance distinct from receipts for downloaded measurements.
    st.markdown('### Modeled relative solar-wind exposure')
    st.write('Flat-surface adaptation of Shukla et al. (2020), Eq. 4: integrate geographic incidence outside an assumed fully shielded 20% of the lunar orbit, then normalize to the equatorial farside maximum. Longitude 0° is the sub-Earth meridian. Values are dimensionless, not measured flux, accumulated dose, or helium-3 concentration. No OMNI data, terrain correction, local magnetic shielding, soil age, retention, or saturation is included.')
    st.write('Select exposure ranking to sort by this index. Balanced ranking and TiO₂/OMAT K-means are unchanged; the model does not fill missing raster measurements. CSV and JSON exports record the model source and shielding assumption.')
    st.markdown('[Model reference: Shukla et al. (2020), Eq. 4](https://doi.org/10.3390/rs12203350)')
    with st.expander('Exposure model assumptions & provenance'):
        st.json(EXPOSURE_MODEL)
    st.markdown('### Product decisions')
    st.write('1. Keep conversation beside the map and table, matching the scoping sketch. Explicit region controls and a visible parsed plan make constraints inspectable.\n\n2. Load only fixed publisher sources and cache successful downloads: source failures show an error instead of substitute data. The workflow stops on unsupported questions instead of silently dropping constraints. These are implementation decisions; user testing remains to be conducted.')
    with st.expander('Loaded source provenance'):
        st.json(st.session_state.provenance)
    st.caption('OMAT uses the publisher-provided 4 ppd scientific overview of the linked 512 ppd product. The map samples every 2 degrees and cannot identify local maxima at full raster resolution. Credit: JAXA/MoonScout; Lemelin et al. (2016).')
    st.markdown('### Sources & setup')
    st.markdown('- [LROC WAC TiO₂ product and calibration](https://data.lroc.im-ldi.com/lroc/view_rdr/WAC_TIO2)\n- [USGS Kaguya OMAT product](https://astrogeology.usgs.gov/search/map/kaguya_lunar_multiband_imager_mi_derived_optical_maturity_omat_50n50s_512ppd_59mpp)\n- [Leafmap Streamlit integration](https://leafmap.org/foliumap/)\n- [OpenAI Responses API](https://developers.openai.com/api/docs/guides/responses)')
    st.info('Questions use the OpenAI Responses API; there is no offline parser. The question and current plan are sent to OpenAI. Ranking and result summaries use deterministic code and real raster samples. Live-model evaluation and real-product calibration validation remain follow-up work.')

## Application tests

Export target: `test_app.py`. 

This file checks that automatic loading and caching work, follow-up queries preserve settings, the interface functions without upload controls, and source failures produce an error without substituting data. It does not establish geological accuracy or validate live LLM performance.

Keep imports and definitions in execution order within this section.


In [ ]:
# export: test_app.py

# Import unittest assertions, failure simulation, the query/data helpers,
# and Streamlit's AppTest runner. Tests use publisher data or explicit errors.
import json
import requests
from types import SimpleNamespace
import unittest
from unittest.mock import patch
from core import DEFAULT, plan_query, PLAN_SCHEMA
# Raster/Streamlit imports are deferred to the integration test class below.
# This keeps numerical and planner checks runnable without native GIS libraries.
# Model tests use coordinates and numerical integration, not invented raster measurements.
import numpy as np
from core import geographic_exposure, rank_sites

In [ ]:
# export: test_app.py


# api_response creates only an API-protocol fixture containing query settings;
# no scientific measurements are generated by these tests.
def api_response(plan=None, supported=True, status='completed', content=None):
    payload = {'status': status, 'output': [{'type': 'message', 'content': content or [
        {'type': 'output_text', 'text': json.dumps({'supported': supported, 'plan': plan or DEFAULT})}]}]}
    return SimpleNamespace(status_code=200, json=lambda: payload)

# Exercise the API boundary without spending API credits or requiring credentials.
class OpenAIPlannerTests(unittest.TestCase):
    # Check follow-up state and the exact structured request sent to OpenAI.
    def test_followup_and_request(self):
        first = dict(DEFAULT, priority='tio2')
        with patch('core.requests.post', side_effect=[api_response(first), api_response(dict(first, top_n=5))]) as post:
            p = plan_query('highest TiO2', DEFAULT, api_key='test-only-key')
            result = plan_query('top 5', p, api_key='test-only-key')
        self.assertEqual(result, dict(first, top_n=5))
        self.assertEqual(DEFAULT['priority'], 'balanced')
        args, kwargs = post.call_args
        self.assertEqual(args[0], 'https://api.openai.com/v1/responses')
        self.assertFalse(kwargs['json']['store'])
        self.assertEqual(kwargs['json']['text']['format']['schema'], PLAN_SCHEMA)
        self.assertIn(json.dumps(first), kwargs['json']['input'][0]['content'])

    # Missing credentials and empty questions must fail before any network call.
    def test_missing_key_and_empty_question(self):
        with patch.dict('os.environ', {}, clear=True), patch('core.OPENAI_API_KEY', ''), patch('core.requests.post') as post:
            for question in ['top 5', ' ']:
                with self.assertRaises(ValueError):
                    plan_query(question, DEFAULT)
            post.assert_not_called()

    # Unsupported requests, invalid bounds, refusals, and truncation cannot become plans.
    def test_invalid_model_results(self):
        responses = [api_response(supported=False), api_response(dict(DEFAULT, west=200)),
                     api_response(status='incomplete'),
                     api_response(content=[{'type': 'refusal', 'refusal': 'declined'}]),
                     api_response(content=[{'type': 'output_text', 'text': 'not JSON'}])]
        for response in responses:
            with self.subTest(response=response), patch('core.requests.post', return_value=response):
                with self.assertRaises(ValueError):
                    plan_query('question', DEFAULT, api_key='test-only-key')

    # Provider errors and timeouts produce bounded errors, never a parser fallback.
    def test_provider_failures(self):
        for code in [401, 403, 429, 500]:
            with self.subTest(code=code), patch('core.requests.post', return_value=SimpleNamespace(status_code=code)):
                with self.assertRaises(ValueError):
                    plan_query('top 5', DEFAULT, api_key='test-only-key')
        with patch('core.requests.post', side_effect=requests.Timeout('private details')):
            with self.assertRaisesRegex(ValueError, 'could not be reached') as caught:
                plan_query('top 5', DEFAULT, api_key='test-only-key')
            self.assertNotIn('private details', str(caught.exception))

# AutomaticSourceTests groups query, real-data cache, UI, and failure-path
# checks. Source-loading tests may download publisher data on a cold cache.
class AutomaticSourceTests(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        # Only the integration suite needs the native raster stack and Streamlit.
        global load_sources, AppTest
        from rasters import load_sources
        from streamlit.testing.v1 import AppTest

    
    # Check real sample counts, TiO2 bounds, all nine download receipts, and
    # equality of the sample table returned by a second cached load.
    def test_sources_and_cache(self):
        data,provenance=load_sources()
        self.assertGreater(len(data),100)
        self.assertTrue(data.tio2.between(0,15).all())
        self.assertEqual(len([p for p in provenance if 'sha256' in p]),9)
        again,_=load_sources()
        self.assertTrue(data.equals(again))
    
    # Run the generated Streamlit app, confirm automatic loading and absence
    # of upload controls, then exercise a query and Methods-page navigation.
    def test_automatic_app(self):
        at=AppTest.from_file(str(__import__('pathlib').Path(__file__).with_name('app.py')),default_timeout=120).run()
        self.assertFalse(at.exception)
        self.assertIsNotNone(at.session_state['data'])
        self.assertEqual(at.sidebar.radio[0].options,['Explore','Methods & project'])
        self.assertEqual(len(at.get('file_uploader')),0)
        at.secrets['OPENAI_API_KEY'] = 'test-only-key'
        at.run()
        self.assertEqual(len(at.toggle), 0)
        at.text_area[0].set_value('highest TiO2')
        [b for b in at.button if b.label=='Find candidates'][0].click()
        with patch('core.requests.post', return_value=api_response(dict(DEFAULT, priority='tio2'))):
            at.run()
        self.assertFalse(at.exception)
        self.assertEqual(at.session_state['plan']['priority'],'tio2')
        # Exercise modeled exposure end-to-end using the real loaded samples.
        at.text_area[0].set_value('highest solar wind exposure')
        [b for b in at.button if b.label=='Find candidates'][0].click()
        with patch('core.requests.post', return_value=api_response(dict(DEFAULT, priority='exposure'))):
            at.run()
        [s for s in at.selectbox if s.label=='Surface layer'][0].set_value('Modeled solar-wind exposure').run()
        self.assertFalse(at.exception)
        data = at.session_state['data']
        self.assertTrue(data.solar_wind_exposure.between(0,1).all())
        ranked = rank_sites(data, at.session_state['plan'])
        self.assertTrue(ranked.solar_wind_exposure.is_monotonic_decreasing)
        self.assertIn('solar_wind_exposure', at.dataframe[0].value.columns)
        self.assertTrue(any('not measured flux' in item[1] for item in at.session_state['history']))
        previous = dict(at.session_state['plan'])
        history = list(at.session_state['history'])
        at.text_area[0].set_value('top 5')
        [b for b in at.button if b.label=='Find candidates'][0].click()
        with patch('core.requests.post', side_effect=requests.Timeout()):
            at.run()
        self.assertEqual(at.session_state['plan'], previous)
        self.assertEqual(at.session_state['history'], history)
        self.assertTrue(any('previous results retained' in e.value for e in at.error))
        # Select GMM in Explore and verify labels and comparison displays render.
        [s for s in at.selectbox if s.label=='Terrain model'][0].set_value('GMM').run()
        self.assertFalse(at.exception)
        self.assertTrue((at.dataframe[0].value['terrain_model']=='GMM').all())
        at.sidebar.radio[0].set_value('Methods & project').run()
        self.assertTrue(any(m.label=='Adjusted Rand agreement' for m in at.metric))
        self.assertFalse(at.exception)
    
    # Simulate a source-loader connection error, then verify an error and retry
    # control appear with no loaded samples or substitute result table.
    def test_source_failure_has_no_fallback(self):
        with patch('rasters.load_sources',side_effect=ConnectionError('Source unavailable')):
            at=AppTest.from_file(str(__import__('pathlib').Path(__file__).with_name('app.py')),default_timeout=120).run()
        self.assertFalse(at.exception)
        self.assertIsNone(at.session_state['data'])
        self.assertEqual(len(at.dataframe),0)
        self.assertTrue(any('could not be loaded' in e.value for e in at.error))
        self.assertTrue(any(b.label=='Retry source loading' for b in at.button))

class GeographicExposureTests(unittest.TestCase):
    # Known physical limits: no shielding recovers cos(latitude), and the
    # equatorial farside stays at one; poles receive zero in this zero-tilt model.
    def test_reference_values_and_symmetry(self):
        self.assertAlmostEqual(float(geographic_exposure(0, 0)), 1-np.sin(.2*np.pi))
        self.assertAlmostEqual(float(geographic_exposure(0, 180)), 1.)
        self.assertAlmostEqual(float(geographic_exposure(60, 180)), .5)
        np.testing.assert_allclose(geographic_exposure([90,-90], 30), 0, atol=1e-15)
        np.testing.assert_allclose(geographic_exposure([0,30,60], [0,45,180], 0),
                                   np.cos(np.deg2rad([0,30,60])))
        np.testing.assert_allclose(geographic_exposure(20, [-180,-90,-30,0,30,90,180]),
                                   geographic_exposure(-20, [180,90,30,0,-30,-90,-180]))
        np.testing.assert_allclose(geographic_exposure(0, [-180,-30,0]),
                                   geographic_exposure(0, [180,330,360]))

    # Independent quadrature checks the piecewise equation against integration
    # of illuminated projected area, excluding the assumed magnetotail arc.
    def test_against_orbital_integration(self):
        phase = np.linspace(-np.pi, np.pi, 200000, endpoint=False) + np.pi/200000
        for f in [0., .14, .20, .27, .5]:
            for lon in [0., 54., 90., 126., 180., -75.]:
                incidence = np.maximum(0., np.cos(np.deg2rad(lon)-phase))
                allowed = np.abs(phase) >= f*np.pi
                expected = np.sum(incidence*allowed) * (2*np.pi/len(phase)) / 2
                self.assertAlmostEqual(float(geographic_exposure(0,lon,f)), expected, places=7)

    # Check branch joins, range, and monotonic shielding across the globe.
    def test_continuity_range_and_shielding(self):
        for boundary in [54.,126.]:
            self.assertAlmostEqual(float(geographic_exposure(0,boundary-1e-7)),
                                   float(geographic_exposure(0,boundary+1e-7)), places=8)
        lat,lon = np.meshgrid(np.linspace(-90,90,181),np.linspace(-180,180,361))
        values = geographic_exposure(lat,lon)
        self.assertTrue(np.all((values >= 0) & (values <= 1)))
        self.assertTrue(np.all(geographic_exposure(lat,lon,.27) <= geographic_exposure(lat,lon,.14)+1e-14))

    def test_invalid_inputs(self):
        for lat,lon,f in [(91,0,.2),(0,np.nan,.2),(np.inf,0,.2),(0,0,-.1),
                          (0,0,.51),(0,0,np.nan),(0,0,True)]:
            with self.subTest(lat=lat,lon=lon,f=f), self.assertRaises(ValueError):
                geographic_exposure(lat,lon,f)

    # Exercise the exposure planner contract without an API call; retrieval is
    # checked against real samples by AutomaticSourceTests above.
    def test_exposure_query_plan(self):
        expected = dict(DEFAULT, priority='exposure')
        with patch('core.requests.post', return_value=api_response(expected)):
            self.assertEqual(plan_query('highest solar wind exposure',DEFAULT,api_key='test-only-key'),expected)


class TerrainComparisonTests(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        # Reuse publisher samples, never invented measurements, for integration
        # and regression checks of both fits. A warm cache needs no network access.
        from rasters import load_sources
        from core import fit_terrain
        cls.raw,_=load_sources()
        cls.fitted,cls.silhouette=fit_terrain(cls.raw)

    def test_memberships_and_identical_inputs(self):
        from sklearn.cluster import KMeans
        from sklearn.preprocessing import StandardScaler
        data=self.fitted
        self.assertTrue(data.attrs['terrain_comparison']['gmm_available'])
        probabilities=data[[f'gmm_p_{i}' for i in range(4)]].to_numpy()
        np.testing.assert_allclose(probabilities.sum(axis=1),1.)
        self.assertTrue(np.all((probabilities>=0)&(probabilities<=1)))
        np.testing.assert_array_equal(probabilities.argmax(axis=1).astype(str),data.gmm_cluster)
        # The original K-means solution and balanced formula must remain intact.
        expected=KMeans(n_clusters=4,random_state=530,n_init=10).fit_predict(StandardScaler().fit_transform(self.raw[['tio2','omat']]))
        np.testing.assert_array_equal(expected.astype(str),data.kmeans_cluster)
        np.testing.assert_allclose(data.score,100*(.65*data.tio2.rank(pct=True)+.35*(-data.omat).rank(pct=True)))

    def test_switching_does_not_rerank_or_mutate(self):
        from core import select_terrain,rank_sites
        km=select_terrain(self.fitted,'K-means')
        gm=select_terrain(self.fitted,'GMM')
        np.testing.assert_array_equal(gm.cluster,gm.gmm_cluster)
        np.testing.assert_array_equal(self.fitted.cluster,self.fitted.kmeans_cluster)
        for priority in ['balanced','tio2','maturity','exposure']:
            plan=dict(DEFAULT,priority=priority)
            self.assertEqual(rank_sites(km,plan).site_id.tolist(),rank_sites(gm,plan).site_id.tolist())

    def test_gmm_failure_preserves_kmeans(self):
        from core import fit_terrain,select_terrain
        # A failed optional fit must not fabricate memberships or disable retrieval.
        with patch('core.GaussianMixture.fit',side_effect=ValueError('test failure')):
            data,_=fit_terrain(self.raw)
        self.assertFalse(data.attrs['terrain_comparison']['gmm_available'])
        self.assertNotIn('gmm_cluster',data)
        select_terrain(data,'K-means')
        with self.assertRaises(ValueError):
            select_terrain(data,'GMM')

    def test_undefined_silhouette(self):
        from core import sampled_silhouette
        # Label-only numerical fixture: a collapsed partition has no silhouette.
        self.assertIsNone(sampled_silhouette(np.zeros((3,2)),[0,0,0],np.arange(3)))

In [ ]:
# export: test_app.py

# api_response creates only an API-protocol fixture containing query settings;
# no scientific measurements are generated by these tests.
def api_response(plan=None, supported=True, status='completed', content=None):
    payload = {'status': status, 'output': [{'type': 'message', 'content': content or [
        {'type': 'output_text', 'text': json.dumps({'supported': supported, 'plan': plan or DEFAULT})}]}]}
    return SimpleNamespace(status_code=200, json=lambda: payload)

# Exercise the API boundary without spending API credits or requiring credentials.
class OpenAIPlannerTests(unittest.TestCase):
    # Check follow-up state and the exact structured request sent to OpenAI.
    def test_followup_and_request(self):
        first = dict(DEFAULT, priority='tio2')
        with patch('core.requests.post', side_effect=[api_response(first), api_response(dict(first, top_n=5))]) as post:
            p = plan_query('highest TiO2', DEFAULT, api_key='test-only-key')
            result = plan_query('top 5', p, api_key='test-only-key')
        self.assertEqual(result, dict(first, top_n=5))
        self.assertEqual(DEFAULT['priority'], 'balanced')
        args, kwargs = post.call_args
        self.assertEqual(args[0], 'https://api.openai.com/v1/responses')
        self.assertFalse(kwargs['json']['store'])
        self.assertEqual(kwargs['json']['text']['format']['schema'], PLAN_SCHEMA)
        self.assertIn(json.dumps(first), kwargs['json']['input'][0]['content'])

    # Missing credentials and empty questions must fail before any network call.
    def test_missing_key_and_empty_question(self):
        with patch.dict('os.environ', {}, clear=True), patch('core.OPENAI_API_KEY', ''), patch('core.requests.post') as post:
            for question in ['top 5', ' ']:
                with self.assertRaises(ValueError):
                    plan_query(question, DEFAULT)
            post.assert_not_called()

    # Unsupported requests, invalid bounds, refusals, and truncation cannot become plans.
    def test_invalid_model_results(self):
        responses = [api_response(supported=False), api_response(dict(DEFAULT, west=200)),
                     api_response(status='incomplete'),
                     api_response(content=[{'type': 'refusal', 'refusal': 'declined'}]),
                     api_response(content=[{'type': 'output_text', 'text': 'not JSON'}])]
        for response in responses:
            with self.subTest(response=response), patch('core.requests.post', return_value=response):
                with self.assertRaises(ValueError):
                    plan_query('question', DEFAULT, api_key='test-only-key')

    # Provider errors and timeouts produce bounded errors, never a parser fallback.
    def test_provider_failures(self):
        for code in [401, 403, 429, 500]:
            with self.subTest(code=code), patch('core.requests.post', return_value=SimpleNamespace(status_code=code)):
                with self.assertRaises(ValueError):
                    plan_query('top 5', DEFAULT, api_key='test-only-key')
        with patch('core.requests.post', side_effect=requests.Timeout('private details')):
            with self.assertRaisesRegex(ValueError, 'could not be reached') as caught:
                plan_query('top 5', DEFAULT, api_key='test-only-key')
            self.assertNotIn('private details', str(caught.exception))

# AutomaticSourceTests groups query, real-data cache, UI, and failure-path
# checks. Source-loading tests may download publisher data on a cold cache.
class AutomaticSourceTests(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        # Only the integration suite needs the native raster stack and Streamlit.
        global load_sources, AppTest
        from rasters import load_sources
        from streamlit.testing.v1 import AppTest

    
    # Check real sample counts, TiO2 bounds, all nine download receipts, and
    # equality of the sample table returned by a second cached load.
    def test_sources_and_cache(self):
        data,provenance=load_sources()
        self.assertGreater(len(data),100)
        self.assertTrue(data.tio2.between(0,15).all())
        self.assertEqual(len([p for p in provenance if 'sha256' in p]),9)
        again,_=load_sources()
        self.assertTrue(data.equals(again))
    
    # Run the generated Streamlit app, confirm automatic loading and absence
    # of upload controls, then exercise a query and Methods-page navigation.
    def test_automatic_app(self):
        at=AppTest.from_file(str(__import__('pathlib').Path(__file__).with_name('app.py')),default_timeout=120).run()
        self.assertFalse(at.exception)
        self.assertIsNotNone(at.session_state['data'])
        self.assertEqual(at.sidebar.radio[0].options,['Explore','Methods & project'])
        self.assertEqual(len(at.get('file_uploader')),0)
        at.secrets['OPENAI_API_KEY'] = 'test-only-key'
        at.run()
        self.assertEqual(len(at.toggle), 0)
        at.text_area[0].set_value('highest TiO2')
        [b for b in at.button if b.label=='Find candidates'][0].click()
        with patch('core.requests.post', return_value=api_response(dict(DEFAULT, priority='tio2'))):
            at.run()
        self.assertFalse(at.exception)
        self.assertEqual(at.session_state['plan']['priority'],'tio2')
        # Exercise modeled exposure end-to-end using the real loaded samples.
        at.text_area[0].set_value('highest solar wind exposure')
        [b for b in at.button if b.label=='Find candidates'][0].click()
        with patch('core.requests.post', return_value=api_response(dict(DEFAULT, priority='exposure'))):
            at.run()
        [s for s in at.selectbox if s.label=='Surface layer'][0].set_value('Modeled solar-wind exposure').run()
        self.assertFalse(at.exception)
        data = at.session_state['data']
        self.assertTrue(data.solar_wind_exposure.between(0,1).all())
        ranked = rank_sites(data, at.session_state['plan'])
        self.assertTrue(ranked.solar_wind_exposure.is_monotonic_decreasing)
        self.assertIn('solar_wind_exposure', at.dataframe[0].value.columns)
        self.assertTrue(any('not measured flux' in item[1] for item in at.session_state['history']))
        previous = dict(at.session_state['plan'])
        history = list(at.session_state['history'])
        at.text_area[0].set_value('top 5')
        [b for b in at.button if b.label=='Find candidates'][0].click()
        with patch('core.requests.post', side_effect=requests.Timeout()):
            at.run()
        self.assertEqual(at.session_state['plan'], previous)
        self.assertEqual(at.session_state['history'], history)
        self.assertTrue(any('previous results retained' in e.value for e in at.error))
        at.sidebar.radio[0].set_value('Methods & project').run()
        self.assertFalse(at.exception)
    
    # Simulate a source-loader connection error, then verify an error and retry
    # control appear with no loaded samples or substitute result table.
    def test_source_failure_has_no_fallback(self):
        with patch('rasters.load_sources',side_effect=ConnectionError('Source unavailable')):
            at=AppTest.from_file(str(__import__('pathlib').Path(__file__).with_name('app.py')),default_timeout=120).run()
        self.assertFalse(at.exception)
        self.assertIsNone(at.session_state['data'])
        self.assertEqual(len(at.dataframe),0)
        self.assertTrue(any('could not be loaded' in e.value for e in at.error))
        self.assertTrue(any(b.label=='Retry source loading' for b in at.button))

class GeographicExposureTests(unittest.TestCase):
    # Known physical limits: no shielding recovers cos(latitude), and the
    # equatorial farside stays at one; poles receive zero in this zero-tilt model.
    def test_reference_values_and_symmetry(self):
        self.assertAlmostEqual(float(geographic_exposure(0, 0)), 1-np.sin(.2*np.pi))
        self.assertAlmostEqual(float(geographic_exposure(0, 180)), 1.)
        self.assertAlmostEqual(float(geographic_exposure(60, 180)), .5)
        np.testing.assert_allclose(geographic_exposure([90,-90], 30), 0, atol=1e-15)
        np.testing.assert_allclose(geographic_exposure([0,30,60], [0,45,180], 0),
                                   np.cos(np.deg2rad([0,30,60])))
        np.testing.assert_allclose(geographic_exposure(20, [-180,-90,-30,0,30,90,180]),
                                   geographic_exposure(-20, [180,90,30,0,-30,-90,-180]))
        np.testing.assert_allclose(geographic_exposure(0, [-180,-30,0]),
                                   geographic_exposure(0, [180,330,360]))

    # Independent quadrature checks the piecewise equation against integration
    # of illuminated projected area, excluding the assumed magnetotail arc.
    def test_against_orbital_integration(self):
        phase = np.linspace(-np.pi, np.pi, 200000, endpoint=False) + np.pi/200000
        for f in [0., .14, .20, .27, .5]:
            for lon in [0., 54., 90., 126., 180., -75.]:
                incidence = np.maximum(0., np.cos(np.deg2rad(lon)-phase))
                allowed = np.abs(phase) >= f*np.pi
                expected = np.sum(incidence*allowed) * (2*np.pi/len(phase)) / 2
                self.assertAlmostEqual(float(geographic_exposure(0,lon,f)), expected, places=7)

    # Check branch joins, range, and monotonic shielding across the globe.
    def test_continuity_range_and_shielding(self):
        for boundary in [54.,126.]:
            self.assertAlmostEqual(float(geographic_exposure(0,boundary-1e-7)),
                                   float(geographic_exposure(0,boundary+1e-7)), places=8)
        lat,lon = np.meshgrid(np.linspace(-90,90,181),np.linspace(-180,180,361))
        values = geographic_exposure(lat,lon)
        self.assertTrue(np.all((values >= 0) & (values <= 1)))
        self.assertTrue(np.all(geographic_exposure(lat,lon,.27) <= geographic_exposure(lat,lon,.14)+1e-14))

    def test_invalid_inputs(self):
        for lat,lon,f in [(91,0,.2),(0,np.nan,.2),(np.inf,0,.2),(0,0,-.1),
                          (0,0,.51),(0,0,np.nan),(0,0,True)]:
            with self.subTest(lat=lat,lon=lon,f=f), self.assertRaises(ValueError):
                geographic_exposure(lat,lon,f)

    # Exercise the exposure planner contract without an API call; retrieval is
    # checked against real samples by AutomaticSourceTests above.
    def test_exposure_query_plan(self):
        expected = dict(DEFAULT, priority='exposure')
        with patch('core.requests.post', return_value=api_response(expected)):
            self.assertEqual(plan_query('highest solar wind exposure',DEFAULT,api_key='test-only-key'),expected)

## Practical limits

- This notebook is an editing and export document, not a Run All analysis notebook. Module-level imports, `__file__`, Streamlit session state and test execution need the generated file contexts.
- Unsaved edits are not exported; a running server does not automatically re-export notebook changes.
- Generated `.py` edits are overwritten. Keep logic in this notebook; the launcher is separate infrastructure.
- Reordering cells changes source order. New circular imports between modules can still fail at runtime even if syntax validation passes.
- One notebook concentrates Git merge conflicts. Clear outputs before committing; do not save secrets in cells or outputs.
- The whole notebook is loaded and validated before export. Replacement is atomic per file, not an all-files transaction; interrupted exports can leave mixed versions. Rerun `--check` before restarting.
- The data cache is outside this notebook. The first load still downloads the approved datasets; no synthetic data is introduced.
